# Pre-Lab 6: Radiometer
Submit one Word/PDF before your section. Complete **Q1–Q6**, including the required molecular Monte Carlo below. Include the two original figures and the two Q6 figures. Run all cells; all numerical inputs are illustrative, not measured bulb properties.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
frames = np.arange(0, 2001, 250)
capture_fps = 1000.0
p = 4
t = (frames-frames[0])/capture_fps
turns = np.arange(len(frames))/p
rpm = 60*(len(frames)-1)/(p*(t[-1]-t[0]))
print("Elapsed seconds:", t[-1], "Revolutions:", turns[-1])
print("Mean rpm:", rpm, "Angular speed rad/s:", rpm*2*np.pi/60)
fig, ax = plt.subplots()
ax.plot(t, turns, "ko")
ax.set(xlabel="Time since first passage (s)", ylabel="Cumulative revolutions")
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig("PreLab6_Passages.png", dpi=300)
plt.show()


## Q1-Q2
Show the worked conversion. Repeat the calculation on paper for 60 fps and for p = 1. State the factor of error and how you will verify the actual capture rate and event count.

In [ ]:
t = np.linspace(0, 15, 301)
final_rpm = 90.0
fig, ax = plt.subplots()
for T in [1.0, 3.0, 6.0]:
    ax.plot(t, final_rpm*(1-np.exp(-t/T)), label=f"T = {T:g} s")
ax.set(xlabel="Time after ideal torque step (s)", ylabel="Rotation speed (rpm)")
ax.legend()
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig("PreLab6_Response.png", dpi=300)
plt.show()


## Q3-Q5
Derive the exponential response and interpret T. Explain the residual-gas mechanism and the limits of lux and rpm. Finish with your measurement plan. In Colab, download the two PNGs from the Files pane and insert them into your report.

## Q6 — Required molecular Monte Carlo
**Before the laboratory:** run this activity and include the requested answers and two figures in your Pre-Lab 6 submission.

**Connection to your experiment:** the lamp creates temperature differences at the vanes. Molecules exchange momentum with those surfaces. We isolate that gas–surface step and compare its sample average with an analytical limit. After the laboratory, use this mechanism to interpret your measured direction, illumination trend and early/late speed changes.

### The algorithm
1. Choose the number of events $N$, gas temperature $T_g$, wall temperatures $T_h,T_c$ [K], and diffuse-reflection fraction $a$.
2. Draw uniform random numbers $U\in(0,1)$ and convert them to **wall-crossing** normal speeds: $v_n=\sqrt{-2k_BT\ln U/m_g}$. Here $k_B$ is Boltzmann's constant [J/K], $m_g$ is molecular mass [kg], and $v_n$ is a positive normal speed [m/s].
3. At a diffuse reflection use the wall temperature; at a specular reflection preserve the incident normal speed magnitude. The wall receives impulse $j_n=m_g(v_{in}+v_{out})$ [kg m/s]. The gas receives the opposite momentum change.
4. Average many events. Multiplying mean impulse by incident molecular flux $\Gamma$ [m$^{-2}$s$^{-1}$] gives wall pressure [Pa]. Subtract cold-face pressure from hot-face pressure.

Uniform random numbers → molecular speeds → collision impulses → mean pressure difference → analytical check.

Both faces receive equal ideal reservoir flux. Paired incident samples cancel in the pressure difference; outgoing diffuse samples are independent. A local random seed makes each run reproducible.

### Assumptions and analytical check
The teaching model has independent free-molecular reservoirs, stationary faces, and no gas–gas collisions, finite edges, enclosure or thermal evolution. It is **not a full DSMC model of the rotor**. Optical black/shiny appearance does not specify molecular reflectivity.

For reservoir pressure $P_g>0$, the expectation is
$$\frac{\Delta P}{P_g}=\frac{a}{2}\left(\sqrt{\frac{T_h}{T_g}}-\sqrt{\frac{T_c}{T_g}}\right).$$
Positive $\Delta P$ means net recoil from hot toward cold. We do not know the actual bulb's pressure or vane temperatures, so we do not convert this into measured rotor torque or rpm.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def molecular_mc(N, Tgas, Thot, Tcold, diffuse_fraction=1.0, seed=402):
    assert int(N) == N and N >= 2
    assert min(Tgas, Thot, Tcold) > 0 and 0 <= diffuse_fraction <= 1
    rng = np.random.default_rng(seed)
    kB = 1.380649e-23
    mg = 28 * 1.66053906660e-27  # illustrative nitrogen; not an identified bulb gas
    def wall_speed(T):
        U = np.maximum(rng.random(N), np.finfo(float).tiny)
        return np.sqrt(-2*kB*T/mg*np.log(U))
    vin = wall_speed(Tgas)
    hot = wall_speed(Thot)
    cold = wall_speed(Tcold)
    diffuse = rng.random(N) < diffuse_fraction
    hot[~diffuse] = vin[~diffuse]
    cold[~diffuse] = vin[~diffuse]
    # Gamma*mg/Pgas, using Pgas = number_density*kB*Tgas.
    factor = 1/np.sqrt(2*np.pi*kB*Tgas/mg)
    samples = factor*(hot-cold)
    exact = diffuse_fraction/2*(np.sqrt(Thot/Tgas)-np.sqrt(Tcold/Tgas))
    return dict(mean=samples.mean(), exact=exact, samples=samples,
                se=samples.std(ddof=1)/np.sqrt(N), hot=hot, cold=cold, kB=kB, mg=mg)

N = 100000
Tgas, Thot, Tcold = 300.0, 330.0, 300.0
a = 1.0
q = molecular_mc(N, Tgas, Thot, Tcold, a, seed=402)
print('Sampled deltaP/Pgas:', q['mean'])
print('Analytical expectation:', q['exact'])
print('Sampling standard error:', q['se'])
assert abs(q['mean']-q['exact']) < 6*max(q['se'], 1e-14)


### Q6a. Molecular velocities
Plot empirical cumulative distributions and their analytical diffuse-wall limits. Why do hotter-wall molecules tend to leave with greater normal speed? Why does a wall-crossing distribution differ from an ordinary bulk Gaussian?


In [ ]:
fig, ax = plt.subplots()
v = np.linspace(0, 1700, 500)
for values, T, color, label in [(q['cold'], Tcold, 'C0', 'Cold wall'),
                                 (q['hot'], Thot, 'C1', 'Hot wall')]:
    ax.plot(np.sort(values), np.arange(1,N+1)/N, color=color, label=label+' MC')
    wall_cdf = 1-np.exp(-q['mg']*v*v/(2*q['kB']*T))
    gas_cdf = 1-np.exp(-q['mg']*v*v/(2*q['kB']*Tgas))
    ax.plot(v, a*wall_cdf+(1-a)*gas_cdf, '--', color=color,
            label=label+' analytical')
ax.set(xlabel='Outgoing normal speed (m/s)', ylabel='Cumulative probability')
ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig('PreLab6_MC_Velocities.png', dpi=250)
plt.show()


### Q6b. Temperature contrast and force direction
Run the sweep below. Include this figure and the velocity-distribution figure in your submission. Describe equal temperatures, a positive temperature contrast, and a reversed hot/cold assignment. Change $a$ to 0.5 and then 0, rerun, and explain the change. Edit a in the parameter cell, then rerun the molecular cells; the default is a = 1.


In [ ]:
contrast = np.arange(0., 61., 5.)
sweep_fraction = a
exact = sweep_fraction/2*(np.sqrt((300+contrast)/300)-1)
fig, ax = plt.subplots()
for j, seed in enumerate([402, 403, 404]):
    sampled = [molecular_mc(N, 300., 300.+dT, 300., sweep_fraction, seed+100*i)['mean']
               for i, dT in enumerate(contrast)]
    ax.plot(contrast, sampled, 'o', label=f'MC seed series {seed}')
ax.plot(contrast, exact, 'k-', label='Analytical wall model')
ax.set(xlabel='Hot minus cold wall temperature (K)',
       ylabel='Pressure difference / reservoir pressure')
ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig('PreLab6_MC_Force.png', dpi=250)
plt.show()
for label, h, c in [('equal',300.,300.), ('hot/cold',330.,300.), ('reversed',300.,330.)]:
    r = molecular_mc(N, 300., h, c, a, seed=700)
    print(label, 'MC:', round(r['mean'],6), 'analytical:', round(r['exact'],6))
zero = molecular_mc(2000, 300., 330., 300., 0., seed=701)
assert zero['mean'] == 0 and zero['exact'] == 0


### Q6c. Why the number of samples matters
Each curve below is one independent realization; its points use nested prefix averages. Explain how sampling scatter changes with $N$. For independent finite-variance events, standard error scales approximately as $N^{-1/2}$: reducing it by two typically requires four times as many samples. More samples do not correct a missing physical mechanism. Run this figure and discuss it briefly; only the two Q6a–Q6b figures are required in the pre-lab document.


In [ ]:
counts = np.unique(np.logspace(2, np.log10(N), 35).astype(int))
fig, ax = plt.subplots()
for seed in [410, 411, 412]:
    r = molecular_mc(N, 300., 330., 300., a, seed)
    prefix = np.cumsum(r['samples'])
    ax.semilogx(counts, prefix[counts-1]/counts, '.-', label=f'Realization {seed}')
ax.axhline(r['exact'], color='k', ls='--', label='Analytical expectation')
ax.set(xlabel='Number of collisions (nested prefixes)',
       ylabel='Pressure difference / reservoir pressure')
ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig('PreLab6_MC_Convergence.png', dpi=250)
plt.show()


### Q6d. Connect this to the radiometer
Compare two **illustrative thermal states**, not fitted temperatures from our bulb. In the second state both faces are warmer, but their temperature difference is smaller. Explain why a lamp-on test need not keep accelerating as the device warms. The observed rpm also depends on gas drag and bearing losses, so force and rpm are not interchangeable.


In [ ]:
states = [('Larger thermal contrast', 300., 330., 300.),
          ('Warmer, smaller contrast', 320., 340., 335.)]
for label, g, h, c in states:
    r = molecular_mc(200000, g, h, c, a, seed=800)
    print(label, '| gas/hot/cold K:', g,h,c,
          '| sampled deltaP/Pgas:', round(r['mean'],6),
          '| analytical:', round(r['exact'],6))


## After the laboratory — required report connection
Use your own plots from `Compare_Lab6_Illumination_Runs.m` and the molecular-force figure from Q6. Explain:
1. Which face/direction you observed and how it relates to gas recoil from a hotter toward a colder face.
2. How measured rpm changed across your lux conditions and from early to late in each run.
3. Whether warming could change the thermal contrast, and what other explanations your data allow.
4. Why the wall Monte Carlo does not quantitatively predict your bulb's rpm: internal pressure, vane temperatures, enclosure/edge effects, inertia and losses are not all measured or modeled.

Do not put normalized wall pressure and rpm on the same axis or fit unknown temperatures to force agreement. You complete this connection in the lab report, after collecting data; no laboratory measurements are required in the pre-lab submission.

Research connections: [Baier, Hardt, Shahabi & Roohi](https://doi.org/10.1103/PhysRevFluids.2.033401); [Lotfian & Roohi](https://doi.org/10.1017/jfm.2018.880).
